In [ ]:
import os
repo_url = "https://github.com/aaronjji/rk8391-devbox.git"
if not os.path.exists("/kaggle/working/gave2-challenge"):
    os.system(f"git clone --recurse-submodules {repo_url} /kaggle/working/gave2-challenge")
os.chdir("/kaggle/working/gave2-challenge")
os.system("git log -1 --oneline")
print(os.getcwd())

In [ ]:
os.system("pip install -q albumentations opencv-python-headless scikit-image scipy networkx sknw huggingface_hub safetensors")

In [ ]:
candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-preliminary/GAVE2_preliminary",
    "/kaggle/input/gave2-preliminary/GAVE2_preliminary",
]
os.system("find /kaggle/input -maxdepth 3")
DATA_ROOT = None
for c in candidates:
    if os.path.exists(f"{c}/validation/images"):
        DATA_ROOT = c
        break
assert DATA_ROOT is not None, f"Dataset not found in {candidates}"
print("DATA_ROOT =", DATA_ROOT)

reg_candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-registered/registered",
    "/kaggle/input/gave2-registered/registered",
]
FFA_ROOT = None
for c in reg_candidates:
    if os.path.exists(f"{c}/validation/FFA_A"):
        FFA_ROOT = c
        break
assert FFA_ROOT is not None, f"Registered FFA dataset not found in {reg_candidates}"
print("FFA_ROOT =", FFA_ROOT)

def find_ckpt_dir(name):
    for c in [f"/kaggle/input/datasets/aaronajit/{name}", f"/kaggle/input/{name}"]:
        if os.path.exists(c):
            return c
    raise AssertionError(f"{name} checkpoint dir not found")

TASK1_CKPT_DIR = find_ckpt_dir("gave2-task1-7fold-ckpts")
TASK2_CKPT_DIR = find_ckpt_dir("gave2-task2-7fold-ckpts")
print("TASK1_CKPT_DIR =", TASK1_CKPT_DIR)
print("TASK2_CKPT_DIR =", TASK2_CKPT_DIR)

TASK1_CKPTS = " ".join(f"{TASK1_CKPT_DIR}/fold{i}_final.pth" for i in range(11))
TASK2_CKPTS = " ".join(f"{TASK2_CKPT_DIR}/fold{i}_final.pth" for i in range(11))
print("TASK1_CKPTS:", TASK1_CKPTS)
print("TASK2_CKPTS:", TASK2_CKPTS)

In [ ]:
# 11-fold ensemble inference, Task1 (3ch CFP), with horizontal-flip TTA --
# used for the Task1 submission channel. (Task3's separate 5-fold source pass
# now runs locally instead -- adding it here pushed total kernel runtime past
# Kaggle's ~12h session cap and got the whole run killed with zero output.)
os.system(
    f"python -u src/predict_ensemble.py --task task1 --tta "
    f"--checkpoints {TASK1_CKPTS} "
    f"--images-dir {DATA_ROOT}/validation/images "
    f"--masks-dir {DATA_ROOT}/validation/masks "
    f"--out-dir predictions/task1/validation"
)

In [ ]:
# 7-fold ensemble inference, Task2 (5ch CFP+FFA), with horizontal-flip TTA
os.system(
    f"python -u src/predict_ensemble.py --task task2 --tta "
    f"--checkpoints {TASK2_CKPTS} "
    f"--images-dir {DATA_ROOT}/validation/images "
    f"--masks-dir {DATA_ROOT}/validation/masks "
    f"--ffa-dir {FFA_ROOT}/validation "
    f"--out-dir predictions/task2/validation"
)

In [ ]:
# Task3 and final zip packaging now happen locally instead of on Kaggle -- this
# kernel's job is just the proven-safe 11-fold+TTA Task1/Task2 predictions
# (matches v3's ~11.2h runtime, safely under Kaggle's ~12h session cap).
os.system("ls -la predictions/task1/validation | head -5")
os.system("ls -la predictions/task2/validation | head -5")
os.system("find predictions -type f | wc -l")